# DS 5007 Natural Language Processing, Assignment 2
## Neural NLP Pipeline: a continuation of the BBC Urdu NLP pipeline

**Student:** i25-8023 | **Section:** DS-A

**GitHub repository:** REPO_URL_PLACEHOLDER

Everything below is implemented from scratch in PyTorch (no pretrained models, no Gensim, no HuggingFace,
and no `nn.Transformer` / `nn.MultiheadAttention` / `nn.TransformerEncoder`). The inputs are the three files
produced in Assignment 1: `cleaned.txt` (sentence-per-line, normalised, lemmatised and stemmed), `raw.txt`
(untouched article bodies) and `Metadata.json` (titles, urls, dates).

| Part | Content |
|---|---|
| 0 | setup, data loading, topic labelling of the 250 articles (used by 1.1, 2 and 3) |
| 1 | TF-IDF, PPMI, t-SNE, Skip-gram Word2Vec, analogies, four-condition comparison |
| 2 | silver POS / NER annotation, BiLSTM tagger (softmax for POS, CRF + Viterbi for NER), evaluation, ablations |
| 3 | Transformer encoder topic classifier, attention heatmaps, BiLSTM vs Transformer comparison |

In [ ]:
# imports, seeds, paths and a helper that makes Urdu text render correctly inside matplotlib figures
import os, sys, re, json, math, time, random, copy
from collections import Counter, defaultdict
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import arabic_reshaper
from bidi.algorithm import get_display
from IPython.display import display

sys.path.insert(0, "src")
from a1_tools import (read_cleaned, read_raw, corpus_word_freq, UrduLemmatizer, UrduStemmer,
                      NUM_TOKEN, PUNCT_TOKENS, PROTECTED_WORDS)
import urdu_lexicon as LEX

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(8)
for folder in ("embeddings", "models", "data", "figures"):
    os.makedirs(folder, exist_ok=True)

plt.rcParams.update({"font.family": ["Segoe UI", "Tahoma", "Arial", "DejaVu Sans"], "figure.dpi": 100,
                     "axes.titlesize": 11, "axes.labelsize": 10})
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)


def ur(text):
    """reshape + bidi so that Urdu strings are joined and right-to-left inside matplotlib"""
    return get_display(arabic_reshaper.reshape(str(text)))


def savefig(name):
    plt.savefig(os.path.join("figures", name), dpi=150, bbox_inches="tight")


print("torch", torch.__version__, "| device:", DEVICE, "| threads:", torch.get_num_threads())

In [ ]:
# loading the three Assignment 1 files: cleaned.txt (main corpus), raw.txt (ablation baseline), Metadata.json
clean_docs = read_cleaned("cleaned.txt")
raw_docs = read_raw("raw.txt")
with open("Metadata.json", encoding="utf-8") as f:
    metadata = {m["article_number"]: m for m in json.load(f)}
ARTICLE_IDS = sorted(clean_docs)
assert ARTICLE_IDS == sorted(raw_docs) == sorted(metadata)

summary = pd.DataFrame({
    "file": ["cleaned.txt", "raw.txt"],
    "articles": [len(clean_docs), len(raw_docs)],
    "sentences": [sum(map(len, clean_docs.values())), sum(map(len, raw_docs.values()))],
    "tokens": [sum(len(s) for d in clean_docs.values() for s in d), sum(len(s) for d in raw_docs.values() for s in d)],
    "types": [len({t for d in clean_docs.values() for s in d for t in s}), len({t for d in raw_docs.values() for s in d for t in s})],
})
display(summary)
print("example cleaned sentence:", " ".join(clean_docs[1][0]))
print("example raw sentence    :", " ".join(raw_docs[1][0]))

### Topic labels for the 250 articles (Section 6.1, needed already in 1.1 and Part 2)
`Metadata.json` has no topic field, so every article is assigned to one of the five assignment categories by a
transparent keyword score. The indicative keywords from the handout (election, government, cricket, inflation,
UN, hospital, ...) are expanded with their Urdu equivalents. Score = keyword hits per 1,000 body tokens plus a
bonus of 15 for every keyword in the title (titles are short and very informative). Articles that hit no
keyword at all are human-interest stories and fall back to *Health & Society*.

In [ ]:
# keyword based assignment of every article to one of the five categories
TOPIC_KEYWORDS = {
    "Politics": """انتخاب الیکشن حکومت وزیر پارلیمان اسمبلی سیاسی سیاست جماعت پارٹی اپوزیشن سینیٹ ووٹ عدالت جج سپریم
                   آئین قانون عمران شریف تحریک_انصاف مسلم_لیگ پیپلز_پارٹی بلاول نواز شہباز مریم وزارت اعظم وزیراعلی گورنر کابینہ""",
    "Sports": """کرکٹ میچ ٹیم کھلاڑی سکور رنز وکٹ اننگز ٹیسٹ_میچ کپتان بیٹنگ بولنگ بولر فٹبال گول ٹورنامنٹ سیریز فائنل
                 ورلڈ کپ کھیل اولمپک ہاکی ٹینس میسی ارجنٹینا بلے گیند پی_سی_بی ٹی_ٹوئنٹی ون_ڈے""",
    "Economy": """مہنگائی تجارت بینک جی_ڈی_پی بجٹ معیشت معاشی اقتصادی روپے ڈالر قیمت تیل برآمد درآمد سرمایہ_کاری کمپنی
                  منڈی مارکیٹ قرض آئی_ایم_ایف ٹیکس شرح_سود کاروبار صنعت بیرل پیٹرول""",
    "International": """اقوام_متحدہ معاہدہ خارجہ دو_طرفہ تنازع جنگ ایران اسرائیل امریکہ امریکی چین روس یوکرین غزہ حوثی یمن
                        سعودی افغانستان طالبان سفارتی سفیر عالمی حملہ میزائل فوج فضائی سرحد مذاکرات ٹرمپ نیٹو انڈیا""",
    "Health & Society": """ہسپتال بیماری ویکسین سیلاب تعلیم صحت ڈاکٹر مریض علاج سکول طالب یونیورسٹی خاتون بچہ خاندان شادی
                           فلم اداکار گانا موسیقی سماجی معاشرہ پولیس قتل ماحول بارش موسم زلزلہ وبا دوا کہانی مذہب مرض امراض
                           کینسر نرس طبی حمل""",
}
TOPIC_KEYWORDS = {c: [k.replace("_", " ") for k in v.split()] for c, v in TOPIC_KEYWORDS.items()}
CATEGORIES = list(TOPIC_KEYWORDS)
CAT2ID = {c: i for i, c in enumerate(CATEGORIES)}


def topic_scores(num):
    body = [t for s in clean_docs[num] for t in s]
    joined = " " + " ".join(body) + " "
    title = metadata[num]["title"]
    return {c: 1000 * sum(joined.count(f" {k} ") for k in kws) / len(body) + 15 * sum(title.count(k) for k in kws)
            for c, kws in TOPIC_KEYWORDS.items()}


article_topic = {}
for num in ARTICLE_IDS:
    scores = topic_scores(num)
    best = max(scores, key=scores.get)
    article_topic[num] = best if scores[best] > 0 else "Health & Society"

with open("data/topic_labels.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in article_topic.items()}, f, ensure_ascii=False, indent=1)

topic_counts = Counter(article_topic.values())
display(pd.DataFrame({"articles": [topic_counts[c] for c in CATEGORIES]}, index=CATEGORIES).T)
for c in CATEGORIES:
    examples = [metadata[n]["title"][:70] for n in ARTICLE_IDS if article_topic[n] == c][:2]
    print(f"{c:17s} e.g. {examples}")

# Part 1: Word Embeddings
## 1.1 TF-IDF weighting
Vocabulary = the 10,000 most frequent tokens of `cleaned.txt`; every other token maps to `<UNK>`. Index 0 is
reserved for `<PAD>` (used by the neural models in Parts 2 and 3) and index 1 for `<UNK>`, so all matrices and
embedding tables share one `word2idx` of size 10,002. Documents are the 250 articles.

TF(w, d) = count(w, d) / |d| (length-normalised so long articles do not dominate the per-category averages),
IDF(w) = log[N / (1 + df(w))], TF-IDF = TF x IDF exactly as in the handout. Words that occur in (almost) every
article get IDF <= 0, which automatically pushes function words to the bottom.

In [ ]:
# vocabulary (10k most frequent tokens + <PAD>, <UNK>) and the raw term-document count matrix
VOCAB_SIZE = 10_000
PAD, UNK = "<PAD>", "<UNK>"
clean_counts = Counter(t for d in clean_docs.values() for s in d for t in s)
word2idx = {PAD: 0, UNK: 1}
for w, _ in clean_counts.most_common(VOCAB_SIZE):
    word2idx[w] = len(word2idx)
idx2word = {i: w for w, i in word2idx.items()}
V = len(word2idx)
with open("embeddings/word2idx.json", "w", encoding="utf-8") as f:
    json.dump(word2idx, f, ensure_ascii=False)


def encode(tokens, vocab=None):
    vocab = vocab or word2idx
    return [vocab.get(t, vocab[UNK]) for t in tokens]


covered = sum(c for w, c in clean_counts.items() if w in word2idx)
print(f"types in cleaned.txt: {len(clean_counts)} | vocabulary: {V} (incl. PAD, UNK) | "
      f"token coverage: {100 * covered / sum(clean_counts.values()):.2f}% | rarest kept word count: {clean_counts.most_common(VOCAB_SIZE)[-1][1]}")

N_DOCS = len(ARTICLE_IDS)
rows, cols = [], []
for r, num in enumerate(ARTICLE_IDS):
    ids = encode([t for s in clean_docs[num] for t in s])
    rows += [r] * len(ids)
    cols += ids
td_counts = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(N_DOCS, V))
td_counts.sum_duplicates()
print("term-document matrix:", td_counts.shape, "| non-zeros:", td_counts.nnz,
      "| check: total count equals corpus tokens ->", int(td_counts.sum()) == sum(clean_counts.values()))
probe = ["پاکستان", "کرکٹ", "حکومت"]
display(pd.DataFrame(td_counts[:5][:, [word2idx[w] for w in probe]].toarray().astype(int), columns=probe,
                     index=[f"article {n}" for n in ARTICLE_IDS[:5]]))

In [ ]:
# TF-IDF = TF(w,d) x log(N / (1 + df(w))), saved as embeddings/tfidf_matrix.npy
doc_len = np.asarray(td_counts.sum(axis=1)).ravel()
tf = sp.diags(1.0 / doc_len) @ td_counts
df = np.asarray((td_counts > 0).sum(axis=0)).ravel()
idf = np.log(N_DOCS / (1.0 + df)).astype(np.float32)
tfidf = (tf @ sp.diags(idf)).toarray().astype(np.float32)
np.save("embeddings/tfidf_matrix.npy", tfidf)
print("tfidf_matrix.npy saved:", tfidf.shape, "| words with idf <= 0 (in >= 249 docs):",
      [idx2word[i] for i in np.where((idf <= 0) & (df > 0))[0]][:15])

In [ ]:
# top-10 most discriminative words per topic: mean TF-IDF inside the category minus mean TF-IDF in the other categories
# (a word must occur in at least max(3, 10%) of the category's articles, otherwise one long story dominates small classes)
SKIP_IDS = {word2idx[PAD], word2idx[UNK]} | {word2idx[t] for t in PUNCT_TOKENS | {NUM_TOKEN} if t in word2idx}
doc_topics = np.array([article_topic[n] for n in ARTICLE_IDS])
top_words = {}
for c in CATEGORIES:
    in_c = doc_topics == c
    inside, outside = tfidf[in_c].mean(axis=0), tfidf[~in_c].mean(axis=0)
    score = inside - outside
    spread = (td_counts[np.where(in_c)[0]] > 0).sum(axis=0).A1          # articles of c that contain the word
    score[spread < max(3, 0.1 * in_c.sum())] = -np.inf                 # ignore words living in a single story
    score[list(SKIP_IDS)] = -np.inf
    top_words[c] = [idx2word[i] for i in np.argsort(-score)[:10]]
display(pd.DataFrame(top_words, index=range(1, 11)))

The lists are clean and topical: cricket vocabulary (ٹیم، میچ، کرکٹ، کھلاڑی، رنز) for Sports, oil / prices /
rupees for Economy, Houthi / Saudi / Iran / drone for International, court / Imran Khan / ministers for Politics and
police / women / film for Health & Society. This is also an independent sanity check of the keyword-based
topic labels: the most discriminative words of each class are mostly words that were *not* in its keyword list.

## 1.2 Pointwise Mutual Information (PPMI)
Word-word co-occurrence counts with a symmetric window of k = 5 tokens on each side, inside sentence boundaries,
over the same 10,002-word vocabulary. PPMI(w1, w2) = max{0, log2[P(w1, w2) / (P(w1) P(w2))]} with all
probabilities estimated from the co-occurrence table.

In [ ]:
# co-occurrence counts with window k = 5, PPMI weighting, saved as embeddings/ppmi_matrix.npy
WINDOW = 5
centres, contexts = [], []
for d in clean_docs.values():
    for s in d:
        ids = np.array(encode(s))
        for off in range(1, WINDOW + 1):
            if len(ids) > off:
                centres += [ids[:-off], ids[off:]]
                contexts += [ids[off:], ids[:-off]]
centres, contexts = np.concatenate(centres), np.concatenate(contexts)
cooc = sp.csr_matrix((np.ones(len(centres), dtype=np.float64), (centres, contexts)), shape=(V, V))
cooc.sum_duplicates()


def ppmi_from_counts(counts):
    counts = counts.tocoo()
    total = counts.sum()
    row_p = np.asarray(counts.sum(axis=1)).ravel() / total
    col_p = np.asarray(counts.sum(axis=0)).ravel() / total
    pmi = np.log2((counts.data / total) / (row_p[counts.row] * col_p[counts.col]))
    keep = pmi > 0
    return sp.csr_matrix((pmi[keep].astype(np.float32), (counts.row[keep], counts.col[keep])), shape=counts.shape)


ppmi = ppmi_from_counts(cooc)
np.save("embeddings/ppmi_matrix.npy", ppmi.toarray())
print(f"co-occurrence pairs: {len(centres):,} | non-zero cells: {cooc.nnz:,} | positive PPMI cells: {ppmi.nnz:,} "
      f"({100 * ppmi.nnz / V ** 2:.2f}% dense) | symmetric: {abs(ppmi - ppmi.T).max() < 1e-4}")
print("ppmi_matrix.npy saved:", ppmi.shape, "float32")

In [ ]:
# cosine nearest neighbours helpers (work for dense embeddings and for sparse PPMI rows)
def l2_normalise(m):
    if sp.issparse(m):
        norms = np.sqrt(np.asarray(m.multiply(m).sum(axis=1)).ravel())
        return sp.diags(1.0 / np.maximum(norms, 1e-9)) @ m
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)


def all_similarities(unit, idx):
    v = unit[idx]
    sims = unit @ (v.T if sp.issparse(unit) else v)
    return np.asarray(sims.todense()).ravel() if sp.issparse(sims) else np.asarray(sims).ravel()


MIN_EVAL_COUNT = 5   # like word2vec's default min_count: words seen < 5 times are not offered as neighbours / answers


def candidate_mask(vocab, counts, min_count=MIN_EVAL_COUNT):
    allowed = np.zeros(len(vocab), dtype=bool)
    for w, i in vocab.items():
        allowed[i] = counts.get(w, 0) >= min_count and w not in (PAD, UNK) and w not in PUNCT_TOKENS and w != NUM_TOKEN
    return allowed


def nearest(unit, vocab, inv, word, allowed, k=5):
    sims = all_similarities(unit, vocab[word])
    sims[~allowed] = -np.inf
    sims[vocab[word]] = -np.inf
    top = np.argsort(-sims)[:k]
    return [(inv[i], float(sims[i])) for i in top]


def neighbour_table(unit, vocab, inv, words, allowed, k=5):
    return pd.DataFrame({w: [f"{n} ({s:.2f})" for n, s in nearest(unit, vocab, inv, w, allowed, k)] for w in words if w in vocab},
                        index=range(1, k + 1))


clean_allowed = candidate_mask(word2idx, clean_counts)
print(f"neighbour / answer candidates (count >= {MIN_EVAL_COUNT}): {clean_allowed.sum():,} of {V:,} vocabulary entries")
ppmi_unit = l2_normalise(ppmi)
PPMI_QUERIES = ["پاکستان", "حکومت", "عدالت", "فوج", "کرکٹ", "ایران", "ہسپتال", "تیل", "فلم", "پولیس", "تعلیم", "میزائل"]
display(neighbour_table(ppmi_unit, word2idx, idx2word, PPMI_QUERIES, clean_allowed, k=5))

### t-SNE of the 200 most frequent tokens
Punctuation, `<NUM>` and function words (the A1 stop list plus the closed-class lexicon) carry no topical
meaning and would all collapse into one blob, so the 200 most frequent *content* tokens are used. Their PPMI rows
are reduced to 50 dimensions with truncated SVD and then projected with t-SNE. Each token was assigned to one of nine semantic
categories by hand (topical fields plus verbs and general discourse words).

In [ ]:
# picking the 200 most frequent content tokens and their hand-assigned semantic category
STOP = PROTECTED_WORDS | LEX.POST | LEX.PRON | LEX.CONJ | LEX.DET | LEX.AUX | LEX.NUM_WORDS | LEX.LETTER_NAMES | PUNCT_TOKENS | {NUM_TOKEN}
SEMANTIC = {
    "Politics & Law": """حکومت وزیر عدالت قانون صدر فیصلہ بیان الزام سیاسی اعظم مقدمہ عمران پارلیمان انتخاب سربراہ حکام سرکار
                         وزارت کارروائی درخواست جج آئین رکن تحقیق اختیار سابق اسمبلی قرار عائد""",
    "Security & Conflict": """فوج حملہ جنگ دفاع پولیس حوثی کارروائی میزائل اسرائیل طیارہ فضائی سکیورٹی نشانہ قتل طالبان جہاز
                              خطرہ ہلاک ڈرون حملے تنازع اہلکار گرفتار""",
    "Geography & Nations": """پاکستان انڈیا ایران سعودی عرب امریکہ امریکی چین ملک افغانستان اسلام آباد پنجاب شہر یمن علاقہ دنیا
                              عالم انڈین انگلینڈ سپین ارجنٹینا مغرب بحر جنوب مرکز ریاست لاہور کراچی افغان""",
    "Sports & Entertainment": """ٹیم کرکٹ کھلاڑی میچ ٹیسٹ ورلڈ کپ رنز کھیل فلم اداکار وکٹ کپتان فائنل سیریز گول اننگز گیند میسی
                                 بال کامیاب شکست ریکارڈ کارکردگی کردار کہانی بیٹنگ""",
    "Economy & Media": """تیل قیمت کمپنی روپے لاکھ کار میڈیا سوشل ویڈیو ٹی پوسٹ صارف خبر رپورٹ لکھا کتاب استعمال نقصان""",
    "People & Society": """لوگ خاتون بچہ فرد بیٹا خاندان زندگی گھر شادی عمر عوام مرد انسان ڈاکٹر خان محمد علی احمد بھائی
                           قوم شخص تعلیم والد""",
    "Time": """وقت سال دن برس ماہ ستمبر اگست روز دہائی آغاز تاریخ گذشتہ حالیہ صدی مہینہ ہفتہ آج دوران سنہ""",
    "Verbs & light verbs": """کر کہا کرنے کرتے کہنا ہونے جا کرنا بتایا جانے کرتا کہتے بن آنے دیتے جات کرتی آیا کریں دینے دکھائی
                              آئے بنانے دیکھ رکھنے دیکھا ادا حاصل شامل جاری پیش شروع قائم فراہم پیدا تیار متاثر واقع ظاہر
                              بند سامنا یاد نظر""",
    "Descriptors & discourse": """طور بات بڑا دوسرا مزید نام نیا غیر کوشش تقریبا پہلا کام بار سامنے معاملہ طرح اہم حصہ مقام
                                  واقعہ واپس مکمل تعلق ادارہ عمل انداز نظام سوال بنیاد پورا آخر مشکل دور موقع دعوی جواب
                                  صلاحیت باعث قریب بے آف تنقید دار واضح مسئلہ دوبارہ بین یقین ضرور جگہ گفتگو سطح اثر اتنا
                                  مدد مسلسل معمول خیال موجود""",
}
SEMANTIC_OF = {}
for cat, words in SEMANTIC.items():
    for w in words.split():
        SEMANTIC_OF.setdefault(w, cat)

tsne_words = [w for w, _ in clean_counts.most_common() if w not in STOP and len(w) > 1][:200]
tsne_cats = [SEMANTIC_OF.get(w, "Other / general") for w in tsne_words]
print(Counter(tsne_cats))
print("words left in 'Other / general':", " ".join(w for w, c in zip(tsne_words, tsne_cats) if c == "Other / general"))

In [ ]:
# SVD (50 dims) of the PPMI rows followed by t-SNE, colour coded by semantic category
svd_input = ppmi[[word2idx[w] for w in tsne_words]]
reduced = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(svd_input)
coords = TSNE(n_components=2, perplexity=25, init="pca", random_state=SEED, metric="cosine").fit_transform(reduced)

palette = dict(zip(list(SEMANTIC) + ["Other / general"],
                   ["#d62728", "#7f3c8d", "#1f77b4", "#2ca02c", "#ff7f0e", "#17becf", "#8c564b", "#bcbd22", "#e377c2", "#b0b0b0"]))
fig, ax = plt.subplots(figsize=(13, 9))
for cat, colour in palette.items():
    sel = [i for i, c in enumerate(tsne_cats) if c == cat]
    ax.scatter(coords[sel, 0], coords[sel, 1], c=colour, s=28 if cat != "Other / general" else 14, label=f"{cat} ({len(sel)})",
               alpha=0.9 if cat != "Other / general" else 0.5)
    for i in sel:
        if cat != "Other / general" or i < 60:
            ax.annotate(ur(tsne_words[i]), coords[i], fontsize=8, alpha=0.85, xytext=(2, 2), textcoords="offset points")
ax.set_title("t-SNE of PPMI vectors for the 200 most frequent content tokens (colour = semantic category)")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.legend(loc="best", fontsize=9, title="semantic category")
savefig("tsne_ppmi.png")
plt.show()

Clear clusters appear for **sports** (cricket / match / team / runs / wicket with the football names), for
**security & conflict** (Houthi, missile, aircraft, attack, Saudi, Yemen) and for **politics & law** (court,
decision, case, Imran Khan). Geography terms split by the story they appear in: Iran / Israel / America sit
next to the war cluster while Pakistani provinces sit next to the domestic politics words, which is exactly what
window-based co-occurrence on a 250-article news corpus should give.

## 2. Skip-gram Word2Vec
### 2.1 Implementation
* training stream: sentences of the corpus, punctuation and `<NUM>` removed, frequent words sub-sampled
  (Mikolov et al., keep probability sqrt(t/f) + t/f with t = 1e-4, redrawn every epoch);
* positive pairs: every (centre, context) pair inside a symmetric window of k = 5 tokens;
* noise distribution P_n(w) proportional to f(w)^(3/4), implemented as a 10^6-entry unigram **noise table**
  (as in the original C code), K = 10 noise words per positive pair;
* two separate embedding matrices, V (centre) and U (context), both |V| x d;
* loss L = -log sigma(u_o . v_c) - sum_k log sigma(-u_k . v_c), averaged over the batch;
* Adam, eta = 0.001, batch 1,024 pairs (>= 512), d = 100, 10 epochs (>= 5), loss logged every 250 steps
  (fused Adam kernel for speed on CPU).

In [ ]:
# training-stream builder, noise table and skip-gram pair generator
W2V_EPOCHS, W2V_BATCH, W2V_LR, NEG_K, SUBSAMPLE_T = 10, 1024, 1e-3, 10, 1e-4
NOISE_TABLE_SIZE = 1_000_000
NON_WORD_RE = re.compile(r"^[\W\d_]+$")


def training_stream(docs, vocab):
    """list of id-arrays (one per sentence) without punctuation / numbers / PAD"""
    sentences = []
    for d in docs.values():
        for s in d:
            ids = [vocab.get(t, vocab[UNK]) for t in s if t not in PUNCT_TOKENS and t != NUM_TOKEN and not NON_WORD_RE.match(t)]
            if len(ids) > 1:
                sentences.append(np.array(ids, dtype=np.int64))
    return sentences


def build_noise_table(sentences, vocab_size, power=0.75, size=NOISE_TABLE_SIZE):
    freq = np.bincount(np.concatenate(sentences), minlength=vocab_size).astype(np.float64)
    probs = freq ** power
    probs /= probs.sum()
    table = np.repeat(np.arange(vocab_size), np.round(probs * size).astype(np.int64))
    return torch.from_numpy(table), freq / freq.sum(), probs


def skipgram_pairs(sentences, word_prob, rng, window=WINDOW, t=SUBSAMPLE_T):
    keep_p = np.minimum(1.0, np.sqrt(t / np.maximum(word_prob, 1e-12)) + t / np.maximum(word_prob, 1e-12))
    kept = [s[rng.random(len(s)) < keep_p[s]] for s in sentences]
    kept = [s for s in kept if len(s) > 1]
    ids = np.concatenate(kept)
    sent_id = np.repeat(np.arange(len(kept)), [len(s) for s in kept])
    cs, xs = [], []
    for off in range(1, window + 1):
        same = sent_id[:-off] == sent_id[off:]
        a, b = ids[:-off][same], ids[off:][same]
        cs += [a, b]
        xs += [b, a]
    return np.concatenate(cs), np.concatenate(xs)


clean_stream = training_stream(clean_docs, word2idx)
noise_table, clean_word_prob, noise_probs = build_noise_table(clean_stream, V)
demo_c, demo_x = skipgram_pairs(clean_stream, clean_word_prob, np.random.default_rng(SEED))
print(f"training sentences: {len(clean_stream):,} | stream tokens: {sum(map(len, clean_stream)):,} | "
      f"pairs in one epoch after sub-sampling: {len(demo_c):,} | noise table: {len(noise_table):,} entries")
top_noise = np.argsort(-noise_probs)[:8]
display(pd.DataFrame({"word": [idx2word[i] for i in top_noise], "unigram P(w)": clean_word_prob[top_noise].round(4),
                      "noise P_n(w) ~ f^0.75": noise_probs[top_noise].round(4)}))
print("first skip-gram pairs of sentence 1:", [(idx2word[c], idx2word[x]) for c, x in zip(demo_c[:6], demo_x[:6])])

In [ ]:
# the skip-gram model with separate centre (V) and context (U) matrices and the negative-sampling BCE loss
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.V = nn.Embedding(vocab_size, dim)   # centre vectors v_c
        self.U = nn.Embedding(vocab_size, dim)   # context vectors u_o
        nn.init.uniform_(self.V.weight, -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.U.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, centre, context, negatives):
        v_c = self.V(centre)                                   # (B, d)
        u_o = self.U(context)                                  # (B, d)
        u_k = self.U(negatives)                                # (B, K, d)
        pos = F.logsigmoid((u_o * v_c).sum(-1))                # log sigma(u_o . v_c)
        neg = F.logsigmoid(-torch.bmm(u_k, v_c.unsqueeze(2)).squeeze(2)).sum(-1)   # sum_k log sigma(-u_k . v_c)
        return -(pos + neg).mean()

    def averaged_embeddings(self):
        emb = 0.5 * (self.V.weight + self.U.weight).detach().cpu().numpy()
        emb[0] = 0.0                                           # PAD row
        return emb


def train_word2vec(sentences, vocab_size, dim=100, epochs=W2V_EPOCHS, batch=W2V_BATCH, lr=W2V_LR, k=NEG_K, log_every=250, tag=""):
    set_seed()
    table, word_prob, _ = build_noise_table(sentences, vocab_size)
    model = SkipGramNS(vocab_size, dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, fused=True)
    rng = np.random.default_rng(SEED)
    curve, running, step, t0 = [], [], 0, time.time()
    for epoch in range(1, epochs + 1):
        c, x = skipgram_pairs(sentences, word_prob, rng)
        perm = rng.permutation(len(c))
        c, x = torch.from_numpy(c[perm]), torch.from_numpy(x[perm])
        epoch_losses = []
        for i in range(0, len(c), batch):
            cb, xb = c[i:i + batch].to(DEVICE), x[i:i + batch].to(DEVICE)
            negs = table[torch.randint(len(table), (len(cb), k))].to(DEVICE)
            loss = model(cb, xb, negs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running.append(loss.item())
            epoch_losses.append(loss.item())
            step += 1
            if step % log_every == 0:
                curve.append((step, float(np.mean(running))))
                running = []
        print(f"[{tag}] epoch {epoch:2d} | pairs {len(c):,} | mean loss {np.mean(epoch_losses):.4f} | {time.time() - t0:.0f}s")
    return model, curve


w2v_c3, curve_c3 = train_word2vec(clean_stream, V, dim=100, tag="C3 cleaned d=100")
emb_c3 = w2v_c3.averaged_embeddings()
np.save("embeddings/embeddings_w2v.npy", emb_c3)
print("embeddings_w2v.npy saved:", emb_c3.shape)

In [ ]:
# training loss curve of the main skip-gram model (C3)
steps, losses = zip(*curve_c3)
plt.figure(figsize=(8, 3.8))
plt.plot(steps, losses, color="#1f77b4")
plt.xlabel(f"training step (batch = {W2V_BATCH} pairs)")
plt.ylabel("negative-sampling BCE loss (mean of 250 steps)")
plt.title("Skip-gram training loss on cleaned.txt (d=100, k=5, K=10, Adam 1e-3)")
plt.grid(alpha=0.3)
savefig("w2v_loss_c3.png")
plt.show()

### 2.2 Evaluation: nearest neighbours and analogies
Query words (romanised in the handout): Pakistan = پاکستان, Hukumat = حکومت, Adalat = عدالت, Maeeshat = معیشت,
Fauj = فوج, Sehat = صحت, Taleem = تعلیم, Aabadi = آبادی. Note that the A1 stemmer strips the final ی of most
آبادی occurrences (they became آباد and merge with اسلام آباد), so only 3 tokens of آبادی survive in
`cleaned.txt`; its neighbours are therefore expected to be noisy.

In [ ]:
# top-10 cosine neighbours of the eight required query words (C3 embeddings)
c3_unit = l2_normalise(emb_c3)
QUERY_WORDS = ["پاکستان", "حکومت", "عدالت", "معیشت", "فوج", "صحت", "تعلیم", "آبادی"]
print("corpus frequency of the query words:", {w: clean_counts[w] for w in QUERY_WORDS})
display(neighbour_table(c3_unit, word2idx, idx2word, QUERY_WORDS, clean_allowed, k=10))
print("note: آبادی itself occurs only 3 times; it is still used as a query, but it is never offered as a neighbour.")

In [ ]:
# analogy tests a : b :: c : ?  answered with v(b) - v(a) + v(c), top-3 candidates (a, b, c excluded, count >= 5)
ANALOGIES = [   # 20 tests written before evaluation; the handout asks for 10, all 20 are reported
    ("ایران", "تہران", "افغانستان", "کابل"),       # country : capital
    ("انڈیا", "دہلی", "روس", "ماسکو"),             # country : capital
    ("امریکہ", "واشنگٹن", "برطانیہ", "لندن"),       # country : capital
    ("سعودی", "ریاض", "ایران", "تہران"),           # country : capital
    ("چین", "بیجنگ", "روس", "ماسکو"),              # country : capital
    ("لاہور", "پنجاب", "پشاور", "خیبر"),            # city : province
    ("لاہور", "پنجاب", "کراچی", "سندھ"),            # city : province
    ("انڈیا", "مودی", "امریکہ", "ٹرمپ"),            # country : leader
    ("امریکہ", "ٹرمپ", "روس", "پوتن"),             # country : leader
    ("میسی", "ارجنٹینا", "بابر", "پاکستان"),        # player : national team
    ("کرکٹ", "وکٹ", "فٹبال", "گول"),               # sport : scoring unit
    ("بیٹنگ", "بلے", "بولنگ", "گیند"),              # action : equipment
    ("عدالت", "جج", "ہسپتال", "ڈاکٹر"),            # institution : professional
    ("فلم", "اداکار", "کرکٹ", "کھلاڑی"),            # field : performer
    ("پاکستان", "روپے", "امریکہ", "ڈالر"),          # country : currency
    ("مرد", "عورت", "بھائی", "بہن"),               # gender
    ("شوہر", "بیوی", "والد", "والدہ"),              # gender
    ("اداکار", "اداکارہ", "مرد", "عورت"),           # gender
    ("بھائی", "بہن", "والد", "والدہ"),              # gender
    ("ستمبر", "اگست", "جولائی", "جون"),             # month : previous month
]


def analogy(unit, vocab, inv, a, b, c, allowed, k=3):
    q = unit[vocab[b]] - unit[vocab[a]] + unit[vocab[c]]
    sims = np.asarray(unit @ (q / np.linalg.norm(q))).ravel()
    sims[~allowed] = -np.inf
    sims[[vocab[a], vocab[b], vocab[c]]] = -np.inf
    return [inv[i] for i in np.argsort(-sims)[:k]]


def run_analogies(unit, vocab, inv, tests, allowed):
    rows = []
    for a, b, c, gold in tests:
        if not all(w in vocab for w in (a, b, c, gold)):
            rows.append({"test": f"{a} : {b} :: {c} : ?", "expected": gold, "top-3": "(word missing)", "correct": False})
            continue
        top = analogy(unit, vocab, inv, a, b, c, allowed)
        rows.append({"test": f"{a} : {b} :: {c} : ?", "expected": gold, "top-3": " | ".join(top), "correct": gold in top})
    return pd.DataFrame(rows)


analogy_c3 = run_analogies(c3_unit, word2idx, idx2word, ANALOGIES, clean_allowed)
display(analogy_c3)
print(f"correct (gold in top-3): {analogy_c3['correct'].sum()} / {len(analogy_c3)}  (first 10: {analogy_c3['correct'][:10].sum()} / 10)")

### Four-condition comparison
* **C1** PPMI rows (10,002-dim sparse vectors) from 1.2;
* **C2** skip-gram trained on `raw.txt` (whitespace / punctuation tokenisation only, its own 10k vocabulary);
* **C3** skip-gram on `cleaned.txt`, d = 100 (the model above);
* **C4** C3 with d = 200.

All skip-gram runs use identical hyper-parameters and seeds. **MRR** is computed on 20 hand-labelled related word
pairs (a, b): every vocabulary word is ranked by cosine similarity to a (a itself, PAD and UNK excluded),
RR = 1 / rank(b) and MRR is the mean over the 20 pairs. As everywhere in this part, only words seen at least 5 times
are candidates. All 40 words exist in both the cleaned and the raw vocabulary.

In [ ]:
# training C2 (raw corpus) and C4 (d = 200) with exactly the same settings
raw_counts = Counter(t for d in raw_docs.values() for s in d for t in s)
raw_word2idx = {PAD: 0, UNK: 1}
for w, _ in raw_counts.most_common(VOCAB_SIZE):
    raw_word2idx[w] = len(raw_word2idx)
raw_idx2word = {i: w for w, i in raw_word2idx.items()}
raw_stream = training_stream(raw_docs, raw_word2idx)
print(f"raw corpus: {len(raw_counts):,} types, vocabulary {len(raw_word2idx):,}, stream tokens {sum(map(len, raw_stream)):,}")

w2v_c2, curve_c2 = train_word2vec(raw_stream, len(raw_word2idx), dim=100, tag="C2 raw d=100")
emb_c2 = w2v_c2.averaged_embeddings()
w2v_c4, curve_c4 = train_word2vec(clean_stream, V, dim=200, tag="C4 cleaned d=200")
emb_c4 = w2v_c4.averaged_embeddings()
np.save("embeddings/embeddings_w2v_raw_c2.npy", emb_c2)
np.save("embeddings/embeddings_w2v_d200_c4.npy", emb_c4)

plt.figure(figsize=(8, 3.8))
for name, curve in (("C2 raw, d=100", curve_c2), ("C3 cleaned, d=100", curve_c3), ("C4 cleaned, d=200", curve_c4)):
    s, l = zip(*curve)
    plt.plot(s, l, label=name)
plt.xlabel(f"training step (batch = {W2V_BATCH} pairs)")
plt.ylabel("negative-sampling BCE loss")
plt.title("Skip-gram training loss for conditions C2, C3 and C4")
plt.legend()
plt.grid(alpha=0.3)
savefig("w2v_loss_conditions.png")
plt.show()

In [ ]:
# 20 hand-labelled related word pairs and the MRR / top-5 comparison of C1-C4
MRR_PAIRS = [
    ("کرکٹ", "میچ"), ("کھلاڑی", "ٹیم"), ("وکٹ", "رنز"), ("ایران", "اسرائیل"), ("افغانستان", "طالبان"),
    ("حوثی", "یمن"), ("سعودی", "عرب"), ("عدالت", "جج"), ("ہسپتال", "ڈاکٹر"), ("فلم", "اداکار"),
    ("پولیس", "گرفتار"), ("لاہور", "پنجاب"), ("میزائل", "ڈرون"), ("تیل", "گیس"), ("روپے", "ڈالر"),
    ("وزیر", "اعظم"), ("فوج", "فضائیہ"), ("امریکہ", "واشنگٹن"), ("چین", "بیجنگ"), ("تعلیم", "سکول"),
]
COMPARE_QUERIES = ["پاکستان", "حکومت", "عدالت", "فوج", "تعلیم"]


def mrr(unit, vocab, pairs, allowed):
    rrs = []
    for a, b in pairs:
        sims = all_similarities(unit, vocab[a])
        sims[~allowed] = -np.inf
        sims[vocab[a]] = -np.inf
        rank = int((sims > sims[vocab[b]]).sum()) + 1
        rrs.append(1.0 / rank)
    return float(np.mean(rrs)), rrs


raw_allowed = candidate_mask(raw_word2idx, raw_counts)
CONDITIONS = {
    "C1 PPMI": (ppmi_unit, word2idx, idx2word, clean_allowed),
    "C2 SG raw d=100": (l2_normalise(emb_c2), raw_word2idx, raw_idx2word, raw_allowed),
    "C3 SG cleaned d=100": (c3_unit, word2idx, idx2word, clean_allowed),
    "C4 SG cleaned d=200": (l2_normalise(emb_c4), word2idx, idx2word, clean_allowed),
}
assert all(a in raw_word2idx and b in raw_word2idx and a in word2idx and b in word2idx for a, b in MRR_PAIRS)
cond_rows, rr_table = [], {}
for name, (unit, vocab, inv, allowed) in CONDITIONS.items():
    score, rrs = mrr(unit, vocab, MRR_PAIRS, allowed)
    rr_table[name] = rrs
    ana = run_analogies(unit, vocab, inv, ANALOGIES, allowed) if not sp.issparse(unit) else None
    cond_rows.append({"condition": name, "MRR (20 pairs)": round(score, 3),
                      "pairs with rank 1": sum(r == 1.0 for r in rrs), "pairs in top 10": sum(r >= 0.1 for r in rrs),
                      f"analogies correct /{len(ANALOGIES)}": int(ana["correct"].sum()) if ana is not None else "n/a"})
display(pd.DataFrame(cond_rows).set_index("condition"))
display(pd.DataFrame(rr_table, index=[f"{a} - {b}" for a, b in MRR_PAIRS]).round(3))

for name, (unit, vocab, inv, allowed) in CONDITIONS.items():
    print(f"\n{name}: top-5 neighbours")
    display(neighbour_table(unit, vocab, inv, COMPARE_QUERIES, allowed, k=5))